
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 12: Reducción de Dimensionalidad y Selección de Variables

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Simplificar datasets de alta dimensionalidad preservando la mayor información posible, y aplicar criterios objetivos para seleccionar las variables más relevantes. Continuamos con `load_diabetes` (Sesión 08).

## 🗺️ Tabla de Contenido
1. [Introducción: la maldición de la dimensionalidad](#intro)
2. [PCA: Análisis de Componentes Principales](#pca)
3. [t-SNE: reducción no lineal para visualizar](#tsne)
4. [Métodos de Filtro](#filtro)
5. [Métodos Envolventes: RFE](#rfe)
6. [Métodos Embebidos: Lasso](#embebido)
7. [Ejemplos de aplicación real](#aplicaciones)
8. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Cuando un dataset tiene muchas columnas, pasan cosas raras: las distancias entre puntos pierden significado, el entrenamiento se vuelve lento, y es imposible "ver" los datos (no podemos graficar 10 dimensiones). A esto se le llama la **maldición de la dimensionalidad**. Hoy aprenderás a reducir el número de variables sin perder la información importante.

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd
import numpy as np

datos = load_diabetes(as_frame=True)
df = datos.frame
X = df.drop(columns=["target"])
y = df["target"]
X.head()

<a id="pca"></a>
## 2. PCA: Análisis de Componentes Principales

### 🔬 Teoría técnica
PCA encuentra nuevas variables (**componentes principales**), combinaciones lineales de las originales, ordenadas de mayor a menor varianza capturada:

1. Estandarizar los datos (obligatorio: PCA es sensible a la escala).
2. Calcular la matriz de covarianza.
3. Obtener eigenvectores/eigenvalues.
4. Proyectar los datos sobre los eigenvectores con mayor eigenvalue.

**Hiperparámetro clave:** `n_components`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X_esc = StandardScaler().fit_transform(X)

pca = PCA(n_components=X.shape[1])  # todas las componentes, para ver la varianza acumulada
pca.fit(X_esc)

varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)

plt.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada, marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95% de varianza")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("PCA - Varianza Explicada Acumulada")
plt.legend()
plt.show()

In [ ]:
# Reducimos a 2 componentes para poder visualizar
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_esc)

y_alto = (y > y.median()).astype(int)  # solo para colorear el gráfico

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_alto, cmap="coolwarm", alpha=0.7)
ax.set_xlabel("Componente Principal 1")
ax.set_ylabel("Componente Principal 2")
ax.set_title("Diabetes proyectada en 2D con PCA (color = avance alto/bajo)")
plt.show()

print(f"Varianza explicada por las 2 primeras componentes: {pca_2d.explained_variance_ratio_.sum():.2%}")

### 💪 Fortalezas y debilidades
- **Fortaleza:** reduce dimensiones de forma matemáticamente óptima (maximiza varianza retenida), acelera modelos posteriores.
- **Debilidad:** las nuevas componentes son combinaciones de las originales — pierdes la interpretación directa ("componente 1" no es una variable real).

### 🧠 Resumen para dummies
Usa PCA cuando quieras **reducir y luego modelar** con menos variables, eligiendo el número de componentes que acumule al menos 95% de la varianza.

<a id="tsne"></a>
## 3. t-SNE: Reducción No Lineal para Visualizar

### 🔬 Teoría técnica
A diferencia de PCA (lineal), t-SNE preserva relaciones de **vecindad local**, revelando agrupaciones que PCA a veces no muestra. **Hiperparámetro clave:** `perplexity`. Es computacionalmente costoso y sus ejes no tienen interpretación directa — **solo se usa para visualizar**, nunca como entrada de otro modelo.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_esc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_alto, cmap="coolwarm", alpha=0.7)
axes[0].set_title("PCA (lineal)")
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_alto, cmap="coolwarm", alpha=0.7)
axes[1].set_title("t-SNE (no lineal)")
plt.tight_layout()
plt.show()

### 🧠 Resumen para dummies
"PCA para reducir y modelar, t-SNE solo para mirar" — nunca uses las coordenadas de t-SNE como entrada de un modelo de predicción.

<a id="filtro"></a>
## 4. Métodos de Filtro

### 🔬 Teoría técnica
Seleccionan variables basándose en una métrica estadística simple (correlación con el objetivo, varianza), **sin entrenar ningún modelo**. Son muy rápidos.

In [ ]:
correlacion_con_objetivo = X.corrwith(y).abs().sort_values(ascending=False)
correlacion_con_objetivo

### 🧠 Resumen para dummies
Es el filtro más simple: "quédate con las variables más correlacionadas con lo que quieres predecir".

<a id="rfe"></a>
## 5. Métodos Envolventes: Recursive Feature Elimination (RFE)

### 🔬 Teoría técnica
RFE entrena el modelo real, elimina la variable menos importante, y repite hasta llegar al número de variables deseado. Más preciso que un filtro, pero más costoso computacionalmente.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe.fit(X, y)

variables_seleccionadas_rfe = X.columns[rfe.support_].tolist()
print("Variables seleccionadas por RFE:", variables_seleccionadas_rfe)

<a id="embebido"></a>
## 6. Métodos Embebidos: Lasso

### 🔬 Teoría técnica
Como vimos en la Sesión 08, Lasso (regularización L1) puede llevar coeficientes exactamente a cero durante el propio entrenamiento — selecciona variables "gratis", como efecto secundario de ajustar el modelo.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.5)
lasso.fit(X_esc, y)

variables_seleccionadas_lasso = X.columns[lasso.coef_ != 0].tolist()
print("Variables que Lasso mantuvo (coeficiente != 0):", variables_seleccionadas_lasso)

### 💪 Fortalezas y debilidades (resumen)

| Método | Fortaleza | Debilidad |
|---|---|---|
| Filtro (correlación) | Rapidísimo, independiente del modelo | Ignora interacciones entre variables |
| Envolvente (RFE) | Considera el modelo real, más preciso | Costoso computacionalmente |
| Embebido (Lasso) | Selección "gratis" durante el entrenamiento | Solo aplica a modelos con regularización L1 |

## 🔎 Laboratorio de profundización: qué optimiza PCA

PCA centra los datos y busca direcciones unitarias de máxima varianza:

$$v_1=\arg\max_{\lVert v\rVert=1}\operatorname{Var}(Xv)$$

Estas direcciones son eigenvectores de la matriz de covarianza. PCA no usa descenso por gradiente en la implementación habitual; usa descomposición lineal/SVD.


In [ ]:
# Paso 1: covarianza y descomposición manual
X_manual = np.array([[2., 1.], [3., 2.], [4., 2.], [5., 4.]])
X_centrada = X_manual - X_manual.mean(axis=0)
covarianza = np.cov(X_centrada, rowvar=False)
valores, vectores = np.linalg.eigh(covarianza)
orden = np.argsort(valores)[::-1]
print("Covarianza:\n", covarianza)
print("Varianzas por dirección:", valores[orden])
print("Primera dirección:", vectores[:, orden[0]])


In [ ]:
# Paso 2: inspeccionar propiedades aprendidas por PCA
pca_demo = PCA(n_components=0.95, svd_solver="full")
pca_demo.fit(X_esc)
print("Componentes retenidos:", pca_demo.n_components_)
print("Varianza explicada:", pca_demo.explained_variance_ratio_)
print("Varianza acumulada:", pca_demo.explained_variance_ratio_.cumsum())


### Hiperparámetros y selección

- PCA: `n_components`, `whiten`, `svd_solver`.
- t-SNE: `perplexity`, `learning_rate`, `max_iter`, `init`; sirve para visualización, no para afirmar clusters.
- Selección: `SelectKBest(k=...)`, RFE (`n_features_to_select`, `step`) y Lasso (`alpha`).

PCA crea variables nuevas; selección conserva variables originales. Ajusta cualquier reductor dentro de cada fold cuando forme parte de un modelo predictivo.


<a id="aplicaciones"></a>
## 7. Ejemplos de Aplicación en el Mundo Real

- Visualizar en 2D un dataset de cientos de variables genéticas o de sensores antes de clusterizar.
- Eliminar ruido antes de aplicar KNN o SVM (sensibles a variables irrelevantes).
- Reconocimiento facial con "Eigenfaces" (PCA aplicado a píxeles de imágenes).

<a id="retos"></a>
## 8. Retos de Práctica

### 🥉 Reto Básico
Aplica PCA a `X` (las 10 variables de `load_diabetes`) y grafica la varianza explicada acumulada. ¿Cuántas componentes necesitas para llegar al 90%?

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Reduce `X` a 2 componentes con PCA y a 2 dimensiones con t-SNE, coloreando ambos gráficos según si el avance de la enfermedad (`y`) está por encima o debajo de la mediana. Compara visualmente cuál separa mejor los grupos.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Compara el subconjunto de variables elegido por el método de Filtro (top 5 por correlación), por RFE (5 variables) y por Lasso. Luego entrena una Regresión Lineal (como en la Sesión 08) usando cada subconjunto y con todas las variables, y compara el R² en un conjunto de prueba (`train_test_split`) para las 4 versiones.

In [ ]:
# Tu solución al Reto Avanzado aquí
